# Phase 7 - Comparative Analysis and Conclusions

This final notebook brings everything together. It retrains all four models
(classical SVM plus the three quantum encodings) in one place so the
comparison is fully self-contained, then evaluates the project hypothesis:

> Entangling encodings (ZZ feature map) will produce measurably different
> decision boundaries than non-entangling encodings (angle encoding), but no
> quantum encoding is expected to beat the classical RBF SVM on this small
> dataset.

We compare on three axes from the project summary: **test accuracy**,
**training convergence stability**, and **circuit depth**, plus a visual
comparison of decision boundaries.

In [ ]:
"""07_comparative_analysis.ipynb"""

# Cell 01 - Imports and a small VQC training helper

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from qiskit import transpile
from qiskit.circuit.library import real_amplitudes, z_feature_map, zz_feature_map
from qiskit_machine_learning.algorithms.classifiers import VQC
from qiskit_machine_learning.circuit.library import raw_feature_vector
from qiskit_machine_learning.optimizers import COBYLA
from qiskit_machine_learning.utils import algorithm_globals
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC

import qml_utils as hu

algorithm_globals.random_seed = hu.RANDOM_SEED
np.random.seed(hu.RANDOM_SEED)


def train_vqc(feature_map, ansatz, x_tr, y_tr, x_te, y_te, maxiter=80):
    """Train a VQC and return a dict of results for the comparison table."""
    history = []
    vqc = VQC(
        feature_map=feature_map,
        ansatz=ansatz,
        optimizer=COBYLA(maxiter=maxiter),
        callback=lambda w, v: history.append(v),
    )
    vqc.fit(x_tr, y_tr)
    return {
        "model": vqc,
        "train_acc": vqc.score(x_tr, y_tr),
        "test_acc": vqc.score(x_te, y_te),
        "history": history,
    }

---
**Cell 02 - Prepare the data once.**
We build two views of the same split: the $[0, \pi]$ scaling for angle and ZZ
encodings, and the padded, normalized version for amplitude encoding.

In [ ]:
# Cell 02 - One dataset, two prepared views

x_train, x_test, y_train, y_test = hu.load_moons(n_samples=200, noise=0.20)

# View used by the classical SVM, angle encoding, and ZZ feature map
x_train_ang, x_test_ang, _ = hu.scale_features(
    x_train, x_test, feature_range=(0.0, np.pi)
)

# View used by amplitude encoding
x_train_01, x_test_01, _ = hu.scale_features(x_train, x_test, feature_range=(0.0, 1.0))
x_train_amp = hu.pad_and_normalize(x_train_01, num_amplitudes=4)
x_test_amp = hu.pad_and_normalize(x_test_01, num_amplitudes=4)

print("Data prepared for all four models.")

---
**Cell 03 - Classical baseline.**

In [ ]:
# Cell 03 - Classical RBF SVM baseline

svm = SVC(kernel="rbf", gamma="scale", random_state=hu.RANDOM_SEED)
svm.fit(x_train_ang, y_train)
svm_train_acc = accuracy_score(y_train, svm.predict(x_train_ang))
svm_test_acc = accuracy_score(y_test, svm.predict(x_test_ang))
print(f"RBF SVM test accuracy: {svm_test_acc:.3f}")

---
**Cell 04 - Train all three quantum encodings.**
Each uses the same `real_amplitudes` ansatz and the same optimizer settings,
so the encoding is the only variable. This cell does the heaviest computation
and may take a minute.

In [ ]:
# Cell 04 - Train the three VQCs

ansatz = real_amplitudes(num_qubits=2, reps=2)

print("Training angle-encoding VQC...")
angle = train_vqc(
    z_feature_map(feature_dimension=2, reps=2),
    ansatz,
    x_train_ang,
    y_train,
    x_test_ang,
    y_test,
)

print("Training amplitude-encoding VQC...")
amplitude = train_vqc(
    raw_feature_vector(feature_dimension=4),
    ansatz,
    x_train_amp,
    y_train,
    x_test_amp,
    y_test,
)

print("Training ZZ-feature-map VQC...")
zz = train_vqc(
    zz_feature_map(feature_dimension=2, reps=2),
    ansatz,
    x_train_ang,
    y_train,
    x_test_ang,
    y_test,
)

print("Done.")

---
**Cell 05 - Test accuracy comparison.**
The dashed line marks the classical SVM. According to the hypothesis we do
**not** expect any quantum bar to clear it on this dataset.

In [ ]:
# Cell 05 - Bar chart of test accuracy

names = ["RBF SVM", "Angle", "Amplitude", "ZZ map"]
accs = [svm_test_acc, angle["test_acc"], amplitude["test_acc"], zz["test_acc"]]
colors = ["gray", "tab:blue", "tab:green", "tab:red"]

plt.figure(figsize=(7, 4))
bars = plt.bar(names, accs, color=colors)
plt.axhline(
    svm_test_acc, linestyle="--", color="black", alpha=0.6, label="classical baseline"
)
plt.ylim(0, 1)
plt.ylabel("test accuracy")
plt.title("Test accuracy by model")
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width() / 2, acc + 0.02, f"{acc:.2f}", ha="center")
plt.legend()
plt.show()

---
**Cell 06 - Decision boundaries side by side.**
The clearest way to judge "measurably different boundaries" is to look at
them together. Note especially whether the **angle** and **ZZ** panels differ.

In [ ]:
# Cell 06 - 2x2 grid of decision boundaries

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

hu.plot_decision_boundary(
    svm.predict,
    x_test_ang,
    y_test,
    title=f"RBF SVM ({svm_test_acc:.2f})",
    ax=axes[0, 0],
)
hu.plot_decision_boundary(
    angle["model"].predict,
    x_test_ang,
    y_test,
    title=f"Angle encoding ({angle['test_acc']:.2f})",
    ax=axes[0, 1],
    grid_steps=40,
)
hu.plot_decision_boundary(
    amplitude["model"].predict,
    x_test_01,
    y_test,
    title=f"Amplitude encoding ({amplitude['test_acc']:.2f})",
    ax=axes[1, 0],
    grid_steps=40,
    transform=lambda p: hu.pad_and_normalize(p, num_amplitudes=4),
)
hu.plot_decision_boundary(
    zz["model"].predict,
    x_test_ang,
    y_test,
    title=f"ZZ feature map ({zz['test_acc']:.2f})",
    ax=axes[1, 1],
    grid_steps=40,
)
plt.tight_layout()
plt.show()

---
**Cell 07 - Training convergence comparison.**
Overlaying the optimization curves shows which encodings were easy or hard to
train. A curve that never settles indicates an unstable optimization
landscape, which is a meaningful result on its own.

In [ ]:
# Cell 07 - Overlay the three convergence curves

plt.figure(figsize=(8, 5))
plt.plot(angle["history"], label="Angle")
plt.plot(amplitude["history"], label="Amplitude")
plt.plot(zz["history"], label="ZZ feature map")
plt.xlabel("optimizer iteration")
plt.ylabel("objective value")
plt.title("Training convergence by encoding")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
**Cell 08 - Summary table.**
A single table with accuracy and circuit depth for every model. Circuit depth
is measured after transpiling each encoding-plus-ansatz circuit to a common
gate set.

In [ ]:
# Cell 08 - Build the results table

basis = ["rx", "ry", "rz", "cx"]


def encoding_depth(feature_map, ansatz, sample_params):
    """Transpile encoding + ansatz to a common basis and return depth."""
    circuit = feature_map.compose(ansatz)
    bound = circuit.assign_parameters(sample_params)
    return transpile(bound, basis_gates=basis, optimization_level=0).depth()


n_params_total = {
    "Angle": z_feature_map(2, reps=2).num_parameters + ansatz.num_parameters,
    "Amplitude": raw_feature_vector(4).num_parameters + ansatz.num_parameters,
    "ZZ map": zz_feature_map(2, reps=2).num_parameters + ansatz.num_parameters,
}

depths = {
    "Angle": encoding_depth(
        z_feature_map(2, reps=2), ansatz, np.random.rand(n_params_total["Angle"])
    ),
    "Amplitude": encoding_depth(
        raw_feature_vector(4),
        ansatz,
        x_train_amp[0].tolist() + list(np.random.rand(ansatz.num_parameters)),
    ),
    "ZZ map": encoding_depth(
        zz_feature_map(2, reps=2), ansatz, np.random.rand(n_params_total["ZZ map"])
    ),
}

table = pd.DataFrame(
    {
        "model": ["RBF SVM", "Angle", "Amplitude", "ZZ map"],
        "train_accuracy": [
            svm_train_acc,
            angle["train_acc"],
            amplitude["train_acc"],
            zz["train_acc"],
        ],
        "test_accuracy": [
            svm_test_acc,
            angle["test_acc"],
            amplitude["test_acc"],
            zz["test_acc"],
        ],
        "circuit_depth": [
            np.nan,
            depths["Angle"],
            depths["Amplitude"],
            depths["ZZ map"],
        ],
    }
)
display(table.round(3))

---
## Conclusions

Use the numbers and plots above to answer the project's questions honestly.
Discuss each point with your own results filled in:

**1. Did entanglement matter?**
Compare the angle-encoding and ZZ-feature-map decision boundaries and
accuracies. If they differ, the entangling gates changed how the model
partitions the feature space, supporting the first half of the hypothesis.

**2. Did any quantum model beat the classical SVM?**
On this small, clean dataset the hypothesis predicts no. If your numbers agree,
that is a valid and expected **null result**, not a failure. Quantum advantage
is not automatic, and encoding choices produce measurable, but not
necessarily superior, differences.

**3. What did each encoding cost?**
Relate the circuit depths and convergence curves to the accuracies. A deeper
circuit or an unstable training curve that does not buy better accuracy is an
important practical lesson.

**Key takeaway:** the encoding strategy is a real architectural decision with
measurable consequences, and honest comparison against a classical baseline,
including null results, is what makes this genuine science.